# compare_all_methods - all approaches at a matched budget

Self-contained. For each seed, `fixed_FPR` (POUNDERS) sets the matched revealed-shot budget, then
every other method runs at that same budget:

| folder | method |
|---|---|
| `fixed_fpr/`    | fixed_FPR (POUNDERS, uniform on FPR circuits) - budget anchor |
| `fixed_no_fpr/` | no_FPR (POUNDERS, uniform all circuits) |
| `adaptive_D/`   | adaptive_FPR (POUNDERS, D-optimal) |
| `adaptive_A/`   | adaptive_FPR (POUNDERS, A-optimal) |
| `adaptive_L/`   | adaptive_FPR (POUNDERS, L-optimal) |
| `lm/`           | pyGSTi built-in LM solver (uniform, matched budget) |

Writes `all_methods_summary.csv`. **Needs PyROL -> run in Docker.** Heavy: adaptive-L is slow (gauge-opt
metric) and LM is a separate fit, so start with **one seed**.

In [62]:
import sys
from pathlib import Path
candidates = [
    Path.cwd(), Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next((p.resolve() for p in candidates if (p / "gst_seed_experiment.py").exists()), None)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Could not locate gst_seed_experiment.py")
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))
from gst_seed_experiment import ExperimentConfig, run_one_experiment, GSTProblem
config = ExperimentConfig.from_json(EXPERIMENT_DIR / "experiment_config.json")
print("Experiment dir:", EXPERIMENT_DIR, "| noise_model:", config.noise_model, "| model_kind:", config.model_kind)

Experiment dir: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments | noise_model: lindblad_hard | model_kind: 1Q


In [63]:
# ------------------------- knobs -------------------------
SEEDS       = "101"                          # start with 1 seed; e.g. "101:106" for 101-105
FIXED_FPR_SHOTS = 2500   # shots/circuit for the anchor; fixed_FPR's accounted cost = budget for all others
# LM-only pass: point this at each archive in turn and re-run.
#   _500 -> SEEDS "101:104" | _800 -> "101:113" | _1500 -> "101:104"
RESULTS_DIR = EXPERIMENT_DIR / "all_methods_comparison_withLM2500"
FORCE       = False    # re-run the adaptive criteria even if completed.json exists.
                      # Required after an allocator code change: the skip check only
                      # compares config.json, and code edits do not alter that.
ADAPTIVE_CRITERIA = ["D"]      # [] = run no adaptive arms (LM-only pass); ["D"] for a normal sweep
ADAPTIVE_ONLY = True   # True: load fixed_FPR/no_FPR/LM from disk, re-run ONLY the criteria above
FORCE_LM    = False     # re-run the LM arm even under ADAPTIVE_ONLY (it now saves x_best.npy).
                       # Set back to False once LM has been regenerated.
LM_MODES, LM_MAXITER = "CPTPLND", 800            # pyGSTi LM baseline settings

# ---------------- inline LM fits during the POUNDERS run (off by default) ----------------
# When ON, an LM MLE is fitted DURING each POUNDERS run on exactly the dataset and circuits
# POUNDERS has consumed at that moment -- no redraw, no reconstruction. That data exists only
# in memory, which is why it cannot be done afterwards from disk. Every fit prints a line
# comparing POUNDERS' incumbent against the LM fit, and all of it lands in lm_checkpoints.csv
# in the arm's folder. With the flag off the run is byte-identical to before.
LM_CKPT_ENABLED = True
LM_CKPT_EVERY   = 1        # fit every N POUNDERS iterations. 1 = EVERY iteration.
                           # Set to 0 to use LM_CKPT_COUNT log-spaced fits instead.
LM_CKPT_COUNT   = 12       # only used when LM_CKPT_EVERY == 0
LM_CKPT_METHODS = ("adaptive_fpr",)   # which POUNDERS methods to instrument
LM_CKPT_UNION   = True     # True: fit the union of FPR-selected circuits so far (matches how
                           # accounted_revealed_shots is counted). False: only this iteration's mask.
LM_CKPT_COLD    = False    # ALSO fit from ideal gates every time. Doubles the cost; leave off
                           # when LM_CKPT_EVERY == 1.

# COST. A cold LM fit on ~650 circuits takes 30-110s, so 600 of them would be ~10 hours per arm.
# LM_CKPT_WARM_FROM = "prev_lm" avoids that: consecutive iterations barely change the data, so
# restarting from the previous fit converges in a handful of LM steps. With a small
# LM_CKPT_MAXITER this brings the per-iteration cost down to a few seconds. The first fit of a
# run is necessarily cold and gets LM_CKPT_MAXITER_FIRST.
#   "pounders" - POUNDERS' current iterate x_k     <- DEFAULT. POUNDERS steps, then LM
#                                                     starts from where POUNDERS just landed.
#   "prev_lm"  - previous iteration's LM estimate  (fastest, but the LM then follows its own
#                                                     trajectory rather than POUNDERS')
#   "cold"     - ideal gates every time            (slowest, most independent)
LM_CKPT_WARM_FROM     = "pounders"
LM_CKPT_MAXITER       = 100   # cap once warm-started
LM_CKPT_MAXITER_FIRST = 800   # cap for the first (necessarily cold) fit of a run

In [64]:
import json
from dataclasses import replace, asdict
import numpy as np
import pandas as pd

INF = "mean_gate_entanglement_infidelity_to_truth"

def parse_seeds(spec):
    seeds = []
    for tok in str(spec).split(","):
        tok = tok.strip()
        if not tok: continue
        if ":" not in tok: seeds.append(int(tok)); continue
        parts = [int(v) for v in tok.split(":")]
        start, stop = parts[0], parts[1]; step = parts[2] if len(parts) == 3 else 1
        seeds.extend(range(start, stop, step))
    return sorted(set(seeds))

def _completed(rd): return (rd / "completed.json").exists() and (rd / "summary.json").exists()
def _config_matches(rd, cfg):
    p = rd / "config.json"
    return p.exists() and json.loads(p.read_text()) == json.loads(json.dumps(asdict(cfg)))

def run_or_load(cfg, seed, method, rd, force):
    """POUNDERS methods via run_one_experiment (with skip-if-done)."""
    if _completed(rd) and not force and _config_matches(rd, cfg):
        print(f"SKIP seed={seed} {rd.name}: completed"); return json.loads((rd / "summary.json").read_text())
    print(f"RUN  seed={seed} {rd.name}")
    # tell the LM-checkpoint hook where to stream its output, so an interrupted run still
    # leaves usable data on disk
    globals()["_ckpt_dir"] = rd
    out = run_one_experiment(config=cfg, data_seed=seed, method=method, output_dir=rd)
    # Persist any inline LM checkpoints gathered during this run. No-op when the hook is off
    # or when its cell was never executed.
    _drain = globals().get("_drain_checkpoints")
    if _drain is not None:
        _drain(rd)
    return out

def fit_or_load_lm(cfg, seed, target_shots, lm_dir, force):
    """pyGSTi LM on uniform shots at the matched budget; writes summary.json + lm_trajectory.csv."""
    if (lm_dir / "summary.json").exists() and (lm_dir / "lm_trajectory.csv").exists() and not force:
        print(f"SKIP seed={seed} lm: completed"); return json.loads((lm_dir / "summary.json").read_text())
    print(f"RUN  seed={seed} lm")
    import re, pygsti
    from pygsti.optimize import SimplerLMOptimizer
    SPAM = "mean_spam_vector_l2_error_to_truth"
    prob = GSTProblem(cfg, seed)
    try: prob.base_model.sim = "map"; prob.truth_model.sim = "map"
    except Exception: pass
    per = max(1, round(target_shots / len(prob.circuits)))
    shots = prob.normalize_shots(per)
    dataset = prob.simulate_dataset(shots, seed=9_000_000 + seed)
    data = pygsti.protocols.ProtocolData(prob.design, dataset)
    opt = SimplerLMOptimizer(maxiter=LM_MAXITER, maxfev=LM_MAXITER, tol=1e-6, init_munu="auto", oob_action="reject")
    proto = pygsti.protocols.StandardGST(modes=LM_MODES, target_model=prob.target_model, optimizer=opt, verbosity=0)
    res = proto.run(data, disable_checkpointing=True)   # else it writes ./standard_gst_checkpoints/
    keys = list(res.estimates.keys()); est_key = LM_MODES if LM_MODES in res.estimates else keys[0]
    est = res.estimates[est_key]
    fit = est.models["final iteration estimate"]
    # Persist the fitted model so LM can be reloaded exactly (for diamond distance and any
    # other post-hoc metric) instead of being refitted. Also write config/metadata so the
    # LM directory is self-describing like the POUNDERS arms.
    lm_dir.mkdir(parents=True, exist_ok=True)
    np.save(lm_dir / "x_best.npy", np.asarray(fit.to_vector(), dtype=float))
    (lm_dir / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2, sort_keys=True))
    (lm_dir / "problem_metadata.json").write_text(json.dumps(
        {"data_seed": int(seed), "truth_seed": int(prob.truth_seed), "method": "lm",
         "num_parameters": int(prob.n), "num_circuits": len(prob.circuits)}, indent=2))
    summ, _, _ = prob.aligned_error_metrics(fit, prob.truth_model, "truth")

    # per-max-length-stage convergence trajectory (one GST estimate per max-length)
    mls = list(getattr(cfg, "max_lengths", []))
    stage_keys = sorted([k for k in est.models if re.fullmatch(r"iteration \d+ estimate", k)],
                        key=lambda k: int(k.split()[1]))
    traj = []
    for si, k in enumerate(stage_keys):
        try:
            st, _, _ = prob.aligned_error_metrics(est.models[k], prob.truth_model, "truth")
            traj.append({"stage": si, "max_length": (mls[si] if si < len(mls) else si),
                         INF: float(st[INF]), SPAM: float(st.get(SPAM, float("nan")))})
        except Exception:
            pass

    out = {"data_seed": seed, "method": "lm",
           INF: float(summ[INF]),
           "accounted_revealed_shots": int(shots.sum()), "physical_precomputed_shots": int(shots.sum()),
           "max_shots_per_circuit": int(shots.max()), "min_shots_per_circuit": int(shots.min()),
           "mean_shots_per_circuit": float(shots.mean()),
           "total_circuits": int(len(prob.circuits)), "revealed_circuits": int(len(prob.circuits)),
           "flag": "LM"}
    lm_dir.mkdir(parents=True, exist_ok=True)
    (lm_dir / "summary.json").write_text(json.dumps(out, indent=2))
    pd.DataFrame(traj).to_csv(lm_dir / "lm_trajectory.csv", index=False)
    return out

def _row(label, seed, s):
    return {"seed": seed, "method": label, INF: s.get(INF, float("nan")),
            "accounted_revealed_shots": s.get("accounted_revealed_shots"),
            "max_shots_per_circuit": s.get("max_shots_per_circuit"), "flag": s.get("flag")}

In [65]:
# ---- inline LM fits: run an MLE mid-POUNDERS on POUNDERS' own data ----
# Installs gst_seed_experiment.LM_CHECKPOINT_HOOK when LM_CKPT_ENABLED is True, clears it
# otherwise. With it cleared this cell has no effect on anything.
#
# Why inline: POUNDERS' dataset at iteration k exists only in memory. final_shots_per_circuit
# .npy is the endpoint, and optimizer_progress.csv keeps cumulative scalars, not the per-circuit
# allocation -- so a post-hoc polish has to REDRAW, which turns polish-vs-POUNDERS into a
# comparison of two different realisations. Fitting here removes that confound entirely.
import time, importlib, inspect
import gst_seed_experiment as _gse

# ---- make sure the RUNNING module actually has the hook call site ----------------------
# The hook is a no-op unless gst_seed_experiment._run_pounders contains the call to
# LM_CHECKPOINT_HOOK. If the kernel imported the module before that line was added, setting
# the attribute here does nothing: the old _run_pounders closure never reads it, and you get
# a normal-looking run with [SHOTS] lines but no [LM-CKPT] lines. Reload and re-bind the
# names cell 1 pulled in by value, so a kernel restart is not required.
def _has_call_site(mod):
    try:
        return "LM_CHECKPOINT_HOOK" in inspect.getsource(mod._run_pounders)
    except Exception:
        return False

if not _has_call_site(_gse):
    _gse = importlib.reload(_gse)
    for _n in ("run_one_experiment", "ExperimentConfig", "GSTProblem"):
        if hasattr(_gse, _n):
            globals()[_n] = getattr(_gse, _n)
    # config was built from the pre-reload class; rebuild it so everything is consistent
    try:
        globals()["config"] = _gse.ExperimentConfig.from_json(
            EXPERIMENT_DIR / "experiment_config.json")
    except Exception as _exc:
        print("  (could not rebuild `config` after reload:", repr(_exc), ")")
    print("reloaded gst_seed_experiment and re-bound run_one_experiment / ExperimentConfig / GSTProblem")

print("gst_seed_experiment:", _gse.__file__)
if not _has_call_site(_gse):
    raise RuntimeError(
        "gst_seed_experiment._run_pounders has no LM_CHECKPOINT_HOOK call site even after a "
        "reload, so inline LM fits cannot run. The file this kernel is importing is shown "
        "above -- if you are in Docker, that path is probably a stale copy rather than a "
        "bind-mount of your edited file. Sync it in, then restart the kernel.")
print("hook call site: present")

_ckpt_rows = []          # the active run's rows; also streamed to disk as they are produced
_ckpt_state = {}
_ckpt_dir = None         # set by run_or_load before each POUNDERS run
_ckpt_x = {"iteration": [], "pounders": [], "lm": []}   # parameter vectors, saved alongside

def _ckpt_flush(final=False):
    """Stream what we have to disk. Called after every row, so killing the run mid-way
    still leaves a complete record up to that point."""
    if _ckpt_dir is None or not _ckpt_rows:
        return
    d = Path(_ckpt_dir); d.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(_ckpt_rows).to_csv(d / "lm_checkpoints.csv", index=False)
    # parameter vectors: 72 floats per point, so the whole run is a few hundred KB. Saving
    # them means any later metric (diamond distance, per-gate breakdown) can be recomputed
    # without redoing a single fit.
    if _ckpt_x["iteration"]:
        # rewrite every row: ~350 KB compressed for a 600-iteration run, a few ms, and it
        # means an interrupted run keeps its vectors instead of losing up to 10 of them
        np.savez_compressed(d / "lm_checkpoint_params.npz",
                            iteration=np.asarray(_ckpt_x["iteration"], dtype=int),
                            x_pounders=np.asarray(_ckpt_x["pounders"], dtype=float),
                            x_lm=np.asarray(_ckpt_x["lm"], dtype=float))

def _revealed_circuits(prob, mask, union_store):
    """Circuit indices behind an FPR residual mask, optionally unioned over the run so far."""
    if mask is None:
        idx = set(range(len(prob.circuits)))
    else:
        rows_ = np.flatnonzero(np.asarray(mask, dtype=bool).reshape(-1))
        idx = set((rows_ // int(prob.outcomes_per_circuit)).tolist())
    if LM_CKPT_UNION:
        union_store |= idx
        idx = set(union_store)
    return sorted(idx)

def _fit_lm_on_current_data(prob, circuit_idx, start_x, maxiter):
    """LM MLE on prob.dataset restricted to `circuit_idx`, started at `start_x`."""
    from pygsti.protocols import (GST, GSTInitialModel, GSTObjFnBuilders, ProtocolData,
                                  GateSetTomographyDesign)
    from pygsti.optimize import SimplerLMOptimizer
    keep = {prob.circuits[i] for i in circuit_idx}
    # keep the max-length ladder nested, restricted to the revealed circuits
    lists = [[c for c in stage if c in keep] for stage in prob.design.circuit_lists]
    if not lists or not lists[-1]:
        return None
    src = prob.processor_spec if prob.processor_spec is not None else prob.target_model
    design = GateSetTomographyDesign(src, lists)
    data = ProtocolData(design, prob.dataset)      # <- the LIVE dataset, not a redraw
    opt = SimplerLMOptimizer(maxiter=maxiter, maxfev=maxiter, tol=1e-6,
                             init_munu="auto", oob_action="reject")
    proto = GST(GSTInitialModel(prob.copy_model_at_x(start_x)), gaugeopt_suite=None,
                objfn_builders=GSTObjFnBuilders.create_from("logl", always_perform_mle=True,
                                                            only_perform_mle=True),
                optimizer=opt, badfit_options=None, verbosity=0, name="ckpt")
    res = proto.run(data, disable_checkpointing=True)
    return res.estimates["ckpt"].models["final iteration estimate"]

def _objective(prob, x, circuits):
    """2*deltaLogL of model `x` against prob.dataset on `circuits`. Lower = better fit.
    Gauge-invariant, unlike infidelity, so it says which point the DATA prefers."""
    try:
        from pygsti.tools import two_delta_logl
        return float(two_delta_logl(prob.copy_model_at_x(x), prob.dataset, circuits))
    except Exception:
        return float("nan")

def _lm_checkpoint_hook(*, state, problem, config, method, iteration,
                        cumulative_shots, running_shots, **_):
    if method not in LM_CKPT_METHODS:
        return
    st = _ckpt_state.setdefault((method, id(problem)),
                                {"union": set(), "plan": None, "done": set(), "prev_x": None})
    idx = _revealed_circuits(problem, state.get("fpr_mask"), st["union"])

    # which iterations get a fit
    if LM_CKPT_EVERY:
        due = (iteration % int(LM_CKPT_EVERY) == 0)
    else:
        if st["plan"] is None:
            n = int(getattr(config, "nfmax", 600))
            st["plan"] = sorted(set(int(round(v)) for v in
                                    np.unique(np.geomspace(1, max(n - 1, 2), LM_CKPT_COUNT))))
            print(f"[LM-CKPT] {method}: fitting at iterations {st['plan']}")
        due = iteration in st["plan"]
    budget = int(getattr(config, "adaptive_total_shot_budget", 0) or 0)
    exhausted = bool(budget) and cumulative_shots >= budget and "exh" not in st["done"]
    if exhausted:
        st["done"].add("exh")
    if not due and not exhausted:
        return

    x_k = np.asarray(state["x"], dtype=float).reshape(-1)
    # `arm`, not `method`: analyze_all_methods inserts its own `method` label column
    rec = {"arm": method, "iteration": int(iteration),
           "cumulative_shots": int(cumulative_shots),
           "shots_on_revealed": int(np.sum(running_shots[idx])),
           "revealed_circuits": len(idx), "at_budget_exhaustion": bool(exhausted)}
    p_summ, _, _ = problem.aligned_error_metrics(problem.copy_model_at_x(x_k),
                                                 problem.truth_model, "truth")
    rec["pounders_" + INF] = float(p_summ[INF])
    rec["pounders_spam"] = float(p_summ.get("mean_spam_vector_l2_error_to_truth", np.nan))
    # objective on exactly the data being fitted, so POUNDERS and LM are directly comparable
    circs_now = [problem.circuits[i] for i in idx]
    rec["pounders_2dlogl"] = _objective(problem, x_k, circs_now)

    # main fit: warm from the previous LM estimate by default, so an every-iteration sweep
    # costs a few seconds per step instead of a minute
    if LM_CKPT_WARM_FROM == "prev_lm" and st["prev_x"] is not None:
        start, maxit, how = st["prev_x"], LM_CKPT_MAXITER, "prev"
    elif LM_CKPT_WARM_FROM == "pounders":
        start, maxit, how = x_k, LM_CKPT_MAXITER, "pounders"
    else:
        start, maxit, how = np.zeros(problem.n), LM_CKPT_MAXITER_FIRST, "cold"
    rec["lm_started_from"] = how

    fits = [("lm", start, maxit)]
    if LM_CKPT_COLD:
        fits.append(("lmcold", np.zeros(problem.n), LM_CKPT_MAXITER_FIRST))
    for tag, x0, mi in fits:
        t0 = time.time()
        try:
            fit = _fit_lm_on_current_data(problem, idx, x0, mi)
            if fit is None:
                continue
            xv = np.asarray(fit.to_vector(), dtype=float)
            if tag == "lm":
                st["prev_x"] = xv          # chain the warm start
                _ckpt_x["iteration"].append(int(iteration))
                _ckpt_x["pounders"].append(x_k.copy())
                _ckpt_x["lm"].append(xv.copy())
            s, _, _ = problem.aligned_error_metrics(fit, problem.truth_model, "truth")
            rec[f"{tag}_" + INF] = float(s[INF])
            rec[f"{tag}_spam"] = float(s.get("mean_spam_vector_l2_error_to_truth", np.nan))
            rec[f"{tag}_2dlogl"] = _objective(problem, xv, circs_now)
            rec[f"{tag}_seconds"] = round(time.time() - t0, 1)
        except Exception as exc:
            print(f"[LM-CKPT] {tag} fit failed at iter {iteration}: {exc!r}")
    _ckpt_rows.append(rec)
    _ckpt_flush()          # stream to disk immediately -- an interrupted run keeps its data

    p, l = rec["pounders_" + INF], rec.get("lm_" + INF)
    po, lo = rec.get("pounders_2dlogl", float("nan")), rec.get("lm_2dlogl", float("nan"))
    print(f"[LM-CKPT] it {iteration:>4} | {len(idx):>4} circ | {rec['shots_on_revealed']:>9,} sh | "
          f"POUNDERS obj {po:>11,.1f} infid {p:.4e}"
          + (f" | LM obj {lo:>11,.1f} infid {l:.4e} | ratio {l/p:6.3f}x | "
             f"{rec.get('lm_seconds', 0):>5.1f}s ({how})" if l is not None else ""))

def _drain_checkpoints(out_dir):
    """Final write + summary for the arm that just finished, then reset for the next one."""
    global _ckpt_rows, _ckpt_x
    if not _ckpt_rows:
        return
    globals()["_ckpt_dir"] = out_dir
    _ckpt_flush(final=True)
    d = pd.DataFrame(_ckpt_rows)
    print(f"  [LM-CKPT] wrote {len(d)} rows -> {Path(out_dir).name}/lm_checkpoints.csv"
          + (f" + lm_checkpoint_params.npz ({len(_ckpt_x['iteration'])} vectors)"
             if _ckpt_x["iteration"] else ""))
    if ("lm_" + INF) in d.columns:
        r = (d["lm_" + INF] / d["pounders_" + INF]).dropna()
        if len(r):
            print(f"  [LM-CKPT] LM/POUNDERS over the run: median {r.median():.3f}x  "
                  f"min {r.min():.3f}x  max {r.max():.3f}x  "
                  f"(LM better on {(r < 1).sum()}/{len(r)})")
        if "lm_seconds" in d:
            print(f"  [LM-CKPT] fit time: total {d['lm_seconds'].sum()/60:.1f} min, "
                  f"median {d['lm_seconds'].median():.1f}s")
    _ckpt_rows = []
    _ckpt_x = {"iteration": [], "pounders": [], "lm": []}
    _ckpt_state.clear()

_gse.LM_CHECKPOINT_HOOK = _lm_checkpoint_hook if LM_CKPT_ENABLED else None
print("inline LM fits:", "ON" if LM_CKPT_ENABLED else "OFF",
      (f"(every={LM_CKPT_EVERY or f'log-spaced x{LM_CKPT_COUNT}'}, "
       f"warm_from={LM_CKPT_WARM_FROM}, maxiter={LM_CKPT_MAXITER}, "
       f"methods={LM_CKPT_METHODS}, union={LM_CKPT_UNION}, cold={LM_CKPT_COLD})")
      if LM_CKPT_ENABLED else "")

gst_seed_experiment: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/gst_seed_experiment.py
hook call site: present
inline LM fits: ON (every=1, warm_from=pounders, maxiter=100, methods=('adaptive_fpr',), union=True, cold=False)


In [66]:
# ---- run all methods per seed; fixed_FPR is the budget anchor ----
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rows = []
for seed in parse_seeds(SEEDS):
    seed_dir = RESULTS_DIR / f"seed_{seed:06d}"; seed_dir.mkdir(parents=True, exist_ok=True)

    # fixed_FPR runs first at FIXED_FPR_SHOTS/circuit; its accounted cost = the matched budget.
    if ADAPTIVE_ONLY and (seed_dir / "fixed_fpr" / "summary.json").exists():
        fixed = json.loads((seed_dir / "fixed_fpr" / "summary.json").read_text())
        print(f"  LOAD seed={seed} fixed_fpr (adaptive-only)")
    else:
        fixed = run_or_load(replace(config, fixed_fpr_shots=int(FIXED_FPR_SHOTS)), seed, "fixed_fpr",
                            seed_dir / "fixed_fpr", FORCE)
    target = int(fixed["accounted_revealed_shots"]); total_circuits = int(fixed["total_circuits"])
    rows.append(_row("fixed_FPR", seed, fixed))
    print(f"  seed={seed}: budget anchor (fixed_FPR accounted) = {target:,}")

    # no_FPR: uniform over ALL circuits at target // total_circuits -> total ~ target
    no_fpr_shots = max(1, target // total_circuits)
    if ADAPTIVE_ONLY and (seed_dir / "fixed_no_fpr" / "summary.json").exists():
        nf = json.loads((seed_dir / "fixed_no_fpr" / "summary.json").read_text())
        print(f"  LOAD seed={seed} no_FPR (adaptive-only)")
    else:
        nf = run_or_load(replace(config, fixed_no_fpr_shots=no_fpr_shots), seed, "fixed_no_fpr",
                         seed_dir / "fixed_no_fpr", FORCE)
    rows.append(_row("no_FPR", seed, nf))

    # adaptive (ADAPTIVE_CRITERIA): acquire up to the anchor budget (lands a few % over -- baseline drag)
    for crit in ADAPTIVE_CRITERIA:
        acfg = replace(config, adaptive_criterion=crit, adaptive_total_shot_budget=target)
        try:
            res = run_or_load(acfg, seed, "adaptive_fpr", seed_dir / f"adaptive_{crit}", FORCE)
            rows.append(_row(f"adaptive_{crit}", seed, res))
        except Exception as exc:
            print(f"  seed={seed} adaptive_{crit}: FAILED {exc!r}")
            rows.append({"seed": seed, "method": f"adaptive_{crit}", INF: float("nan"),
                         "accounted_revealed_shots": 0, "max_shots_per_circuit": 0, "flag": "FAILED"})

    # LM: uniform over all circuits at the anchor budget
    if ADAPTIVE_ONLY and not FORCE_LM and (seed_dir / "lm" / "summary.json").exists():
        lm = json.loads((seed_dir / "lm" / "summary.json").read_text())
        print(f"  LOAD seed={seed} LM (adaptive-only)")
        rows.append(_row("LM", seed, lm))
    else:
        try:
            lm = fit_or_load_lm(config, seed, target, seed_dir / "lm", FORCE or FORCE_LM)
            rows.append(_row("LM", seed, lm))
        except Exception as exc:
            print(f"  seed={seed} LM: FAILED {exc!r}")
            rows.append({"seed": seed, "method": "LM", INF: float("nan"),
                         "accounted_revealed_shots": 0, "max_shots_per_circuit": 0, "flag": "FAILED"})

    pd.DataFrame(rows).to_csv(RESULTS_DIR / "all_methods_summary.csv", index=False)
    print(f"seed {seed} done -> budget ~{target:,}")

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "all_methods_summary.csv", index=False)
print("\nsaved", RESULTS_DIR / "all_methods_summary.csv")

  LOAD seed=101 fixed_fpr (adaptive-only)
  seed=101: budget anchor (fixed_FPR accounted) = 1,265,000
  LOAD seed=101 no_FPR (adaptive-only)
RUN  seed=101 adaptive_D
[POUDERS] Beginning gradient-based optimization.


/usr/local/lib/python3.11/site-packages/pygsti/forwardsims/mapforwardsim.py:732: UserWarning: Generating dense process matrix representations of circuits or gates 
can be inefficient and should be avoided for the purposes of forward 
simulation/calculation of circuit outcome probability distributions 
when using the MapForwardSimulator.
  _warnings.warn('Generating dense process matrix representations of circuits or gates \n'


[POUDERS] FPR reduction selected 336/3836 residuals; union=336/3836; active=336/3836.
[POUDERS] GST model evaluation: Jacobian rows computed=336, reused=0, probabilities reused=False.
[POUDERS] Initial residual/Jacobian evaluation took 0.60 seconds.
[POUDERS] Initial point evaluated.
[POUDERS] nf: 0, full f(x): 215887.96984771488, FPR f(x): 12209.519320949752, selected: 336/3836, union: 336/3836
iter 0: no adaptive top-up (previous rho=unavailable).
[SHOTS] adaptive_fpr iter 0: 252000/1265000 cumulative revealed shots


/usr/local/lib/python3.11/site-packages/pygsti/tools/optools.py:176: UserWarning: 
            Input matrix is not PSD up to tolerance 1.8189894035458565e-12.
            We'll project out the bad eigenspaces to only work with the PSD part.
            
  _warnings.warn(message)
/usr/local/lib/python3.11/site-packages/pygsti/objectivefns/objectivefns.py:4502: RuntimeWarning: divide by zero encountered in divide
  p5over_lsvec = 0.5/lsvec


[LM-CKPT] it    0 |  168 circ |   252,000 sh | POUNDERS obj   114,622.4 infid 2.2977e-03 | LM obj       153.7 infid 1.1669e-03 | ratio  0.508x |   6.6s (pounders)
[POUDERS] Model combine took 0.01 seconds.
[POUDERS] nf: 0, delta: 0.1, full f(x): 215887.96984771488, FPR f(x): 12209.519320949752, selected: 336/3836, union: 336/3836, ng: 175078.74071649928
[POUDERS] Starting PyROL trust-region solve with n=72.
----------------------rol imported------------------

Truncated CG Trust-Region Solver
Trust-Region Model: Kelley-Sachs
  iter  value          gnorm          snorm          delta          #fval     #grad     tr_flag   iterCG    flagCG    
  0     0.000000e+00   6.483378e-01                  1.000000e-01   
  1     -4.636687e+02  5.890721e-01   1.000001e-01   2.500000e-01   7         3         0         19        3         
  2     -5.034726e+02  4.923664e-01   2.500000e-01   6.250000e-01   13        5         0         9         3         
  3     -5.034726e+02  4.923664e-01   2.829

...

/usr/local/lib/python3.11/site-packages/pygsti/forwardsims/mapforwardsim.py:732: UserWarning: Generating dense process matrix representations of circuits or gates 
can be inefficient and should be avoided for the purposes of forward 
simulation/calculation of circuit outcome probability distributions 
when using the MapForwardSimulator.
  _warnings.warn('Generating dense process matrix representations of circuits or gates \n'


[POUDERS] FPR reduction selected 694/3836 residuals; union=1280/3836; active=1280/3836.
iter 38: reused center probabilities/Jacobian for adaptive allocation.
iter 38: N_k=2000, placed 2000 shots over 25/640 circuits (previous rho=0.891834; piloted 0 new; criterion=D, FW gap=1.01e-05).
[SHOTS] adaptive_fpr iter 38: 1036000/1265000 cumulative revealed shots


[LM-CKPT] it   38 |  640 circ | 1,036,000 sh | POUNDERS obj       618.5 infid 6.0845e-05 | LM obj       596.1 infid 7.3798e-05 | ratio  1.213x |  33.8s (pounders)
[POUDERS] GST model evaluation: Jacobian rows computed=0, reused=1280, probabilities reused=True.
[POUDERS] Adaptive-shot hook (iter 38): added 2000 shots; center refreshed.
[POUDERS] Model combine took 0.02 seconds.
[POUDERS] nf: 38, delta: 0.0015625, full f(x): 3740.343719991653, FPR f(x): 1214.1336341924202, selected: 1280/3836, union: 1280/3836, ng: 27325.14453491999
[POUDERS] Starting PyROL trust-region solve with n=72.
----------------------rol imported------------------

Truncated CG Trust-Region Solver
Trust-Region Model: Kelley-Sachs
  iter  value          gnorm          snorm          delta          #fval     #grad     tr_flag   iterCG    flagCG    
  0     0.000000e+00   1.307283e-02                  1.562500e-03   
  1     -4.777589e+00  1.301835e-02   1.562500e-03   3.906250e-03   8         3         0         10

/usr/local/lib/python3.11/site-packages/pygsti/tools/optools.py:176: UserWarning: 
            Input matrix is not PSD up to tolerance 1.8189894035458565e-12.
            We'll project out the bad eigenspaces to only work with the PSD part.
            
  _warnings.warn(message)


  [LM-CKPT] wrote 39 rows -> adaptive_D/lm_checkpoints.csv + lm_checkpoint_params.npz (39 vectors)
  [LM-CKPT] LM/POUNDERS over the run: median 0.220x  min 0.027x  max 1.215x  (LM better on 30/39)
  [LM-CKPT] fit time: total 18.2 min, median 27.6s
  LOAD seed=101 LM (adaptive-only)
seed 101 done -> budget ~1,265,000

saved /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/all_methods_comparison_withLM2500/all_methods_summary.csv


In [67]:
# ---- quick table ----
# `df` is only defined once cell 5 finishes. Fall back to the CSV so this cell also works
# after a kernel restart, or while the sweep is still running (cell 5 rewrites the CSV
# after every seed, so the file is always current).
try:
    df
except NameError:
    df = pd.read_csv(RESULTS_DIR / "all_methods_summary.csv")
    print(f"(df not in memory - loaded {len(df)} rows from {RESULTS_DIR.name}/all_methods_summary.csv)")

print(df[["seed", "method", INF, "accounted_revealed_shots", "max_shots_per_circuit", "flag"]].to_string(index=False))
print("\n=== median infidelity by method ===")
print(df.groupby("method")[INF].median().sort_values().to_string())


 seed     method  mean_gate_entanglement_infidelity_to_truth  accounted_revealed_shots  max_shots_per_circuit          flag
  101  fixed_FPR                                    0.000071                   1265000                   2500     12.806294
  101     no_FPR                                    0.000243                   1263962                    659     38.574617
  101 adaptive_D                                    0.000061                   1036000                   5636  27325.144535
  101         LM                                    0.000046                   1265880                    660            LM

=== median infidelity by method ===
method
LM            0.000046
adaptive_D    0.000061
fixed_FPR     0.000071
no_FPR        0.000243


Now open **`analyze_all_methods.ipynb`** for the comparison plots (grouped bars + shot efficiency).